In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
#Q1 Answer 2
import pandas as pd
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train['label'] = train['answer'].map(label_map)
print(train['label'].iloc[150])

2


In [3]:
#Q2 Answer 407
row = train.iloc[0]
formatted = str(row['prompt']) + " [SEP] " + str(row['B'])
print(len(formatted))

407


In [4]:
#Q3 Answer 5 Can be seen directly from question as 1 for each A-E
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
row = train.iloc[0]
options = ['A', 'B', 'C', 'D', 'E']
formatted = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]
encoded = tokenizer(formatted, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
input_ids = encoded['input_ids'].unsqueeze(0)
print(input_ids.shape)

torch.Size([1, 5, 128])


In [5]:
#Q4 Answer 10240
options = ['A', 'B', 'C', 'D', 'E']
all_encoded = []

for i in range(16):
    row = train.iloc[i]
    formatted = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]
    encoded = tokenizer(formatted, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
    all_encoded.append(encoded['input_ids'])

import torch
input_ids = torch.stack(all_encoded)
print(input_ids.shape)
print(input_ids.numel())

torch.Size([16, 5, 128])
10240


In [6]:
#Q5 Answer 5 Could be directly given from shape 5 logits one for each
from transformers import AutoModelForMultipleChoice
model_mc = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

row = train.iloc[0]
options = ['A', 'B', 'C', 'D', 'E']
formatted = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]

encoded = tokenizer(formatted, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
input_ids = encoded['input_ids'].unsqueeze(0)
attention_mask = encoded['attention_mask'].unsqueeze(0)

with torch.no_grad():
    outputs = model_mc(input_ids=input_ids, attention_mask=attention_mask)

print(outputs.logits.shape)
print(outputs.logits.shape[1])

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 5])
5


In [7]:
#Q6 Answer 0 Could be given directly as scaler tensor has no dimension
label = torch.tensor([train['label'].iloc[0]])
with torch.no_grad():
    outputs = model_mc(input_ids=input_ids, attention_mask=attention_mask, labels=label)
print(outputs.loss)
print(outputs.loss.dim())

tensor(1.5941)
0


In [8]:
#Q7 Answer 
from peft import get_peft_model, LoraConfig, TaskType
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
lora_model = get_peft_model(model_mc, lora_config)
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(trainable_params)

295681


In [9]:
#Q8 Answer 5 could be given from question
from datasets import Dataset
options = ['A', 'B', 'C', 'D', 'E']
def prepare_dataset(df):
    all_input_ids = []
    all_attention_masks = []
    all_labels = []
    for i in range(len(df)):
        row = df.iloc[i]
        formatted = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]
        encoded = tokenizer(formatted, padding='max_length', truncation=True, max_length=128)
        all_input_ids.append(encoded['input_ids'])
        all_attention_masks.append(encoded['attention_mask'])
        all_labels.append(label_map[row['answer']])
    return Dataset.from_dict({
        'input_ids': all_input_ids,
        'attention_mask': all_attention_masks,
        'labels': all_labels
    })
hf_dataset = prepare_dataset(train.iloc[:100])
print(len(hf_dataset[0]['input_ids']))

5


In [10]:
#Q9 Answer 4 could be given from question max_steps
from transformers import TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig, TaskType

train_32 = prepare_dataset(train.iloc[:32])

model_mc2 = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
lora_model2 = get_peft_model(model_mc2, lora_config)

training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1
)

trainer = Trainer(
    model=lora_model2,
    args=training_args,
    train_dataset=train_32
)

trainer.train()
print(trainer.state.global_step)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,1.595141
2,1.677405
3,1.600097
4,1.556641


4


In [11]:
#Q10 Answer 0.2109
import torch.nn.functional as F

row = train.iloc[0]
formatted = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]

encoded = tokenizer(formatted, padding='max_length', truncation=True, max_length=64, return_tensors='pt')
input_ids = encoded['input_ids'].unsqueeze(0)
attention_mask = encoded['attention_mask'].unsqueeze(0)

with torch.no_grad():
    outputs = lora_model2(input_ids=input_ids, attention_mask=attention_mask)

probs = F.softmax(outputs.logits, dim=-1)
print(probs)
print(round(probs[0][4].item(), 4))

tensor([[0.1833, 0.2526, 0.1801, 0.1730, 0.2109]])
0.2109
